# Самостоятельная работа 2 · 12 ноября 2026
## Финансовые данные: загружаем и проверяем сами

Очного занятия нет, работа рассчитана на **80 + 80 минут** с перерывом. Вы сами загрузите историю торгов с Московской биржи, составите паспорт данных и найдёте в них ловушки, о которых говорится в главах 2 и 3 книги. Доходность и риск сегодня не считаем – это тема следующих занятий.

**Что понадобится:** Google Colab или Jupyter с pandas, matplotlib и requests, доступ в интернет. Данные приходят с сервера ISS Московской биржи; ключи и регистрация не нужны.

**Что сдаём:** выполненный ноутбук с ответами в ячейках «**Ответ:**», паспорт данных своей акции и вывод на 150–250 слов. Сдача – через [Google-форму](https://forms.gle/sP7LxYF9y8hMVgPK7).

**Имя и группа:** …

**Моя акция:** … (выберите одну из списка в задании 1)

### Первая часть · 80 минут
| Минуты | Что делаем |
|---|---|
| 20 | Читаем [главу 2](https://quantviews.github.io/quant-finance-book/02-data-sources.html): разделы «Какие бывают финансовые данные» и «Московская биржа и ISS» |
| 10 | Задание 1. Карточка бумаги |
| 10 | Задание 2. Сколько строк вернул запрос |
| 25 | Задание 3. Загружаем всю историю |
| 15 | Задание 4. Паспорт данных |

### Вторая часть · 80 минут
| Минуты | Что делаем |
|---|---|
| 15 | Читаем [главу 3](https://quantviews.github.io/quant-finance-book/03-data-preparation.html): разделы «Сплиты и консолидации» и «Дивиденды и скорректированная цена» |
| 20 | Задание 5. Консолидация акций ВТБ |
| 15 | Задание 6. Дивидендный разрыв |
| 15 | Задание 7. Самые большие движения вашей акции |
| 15 | Задание 8. Вывод и журнал ИИ |

Если застряли больше чем на 10 минут, запишите в ячейку ответа, что пробовали, текст ошибки и конкретный вопрос, и переходите к следующему заданию. Непонятные строки кода можно разобрать по [приложению A](https://quantviews.github.io/quant-finance-book/appendix-python.html).

## Подготовка
Ячейка ниже подключает библиотеки и задаёт функцию `iss_get`. Она отправляет запрос к ISS и превращает каждую таблицу ответа в `DataFrame`. Прочитайте её: ответ ISS – это набор таблиц, у каждой есть список столбцов `columns` и строки `data` (глава 2).

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

ISS = "https://iss.moex.com/iss"

def iss_get(path, **params):
    """Запрос к ISS. Возвращает словарь: имя таблицы -> DataFrame."""
    params = {"iss.meta": "off", **params}
    resp = requests.get(f"{ISS}/{path}.json", params=params, timeout=30)
    resp.raise_for_status()
    return {name: pd.DataFrame(t["data"], columns=t["columns"])
            for name, t in resp.json().items()}

print("Готово")

## Задание 1. Карточка бумаги · 10 минут
Выберите **одну** акцию и впишите её код в `TICKER`:

GAZP, LKOH, GMKN, ROSN, NVTK, TATN, MGNT, MTSS, PLZL, CHMF, NLMK, MOEX, ALRS, AFLT, PHOR, SNGS, IRAO, HYDR, RUAL, SIBN, MAGN, PIKK, SBERP, TRNFP, FEES, RTKM, BSPB.

SBER и VTBR не берите: они понадобятся в заданиях 5–6.

Запросите карточку бумаги: путь `securities/{TICKER}`. В ответе две таблицы: `description` (параметры выпуска) и `boards` (режимы торгов).

Ответьте по таблицам: полное название и ISIN; число акций в выпуске (`ISSUESIZE`); уровень листинга (`LISTLEVEL`); в каких режимах бумага торгуется или торговалась; какой режим основной (`is_primary`) и с какой даты в нём есть история (`history_from`).

**Ответ:** …

In [ ]:
TICKER = "..."   # ваш код акции
card = iss_get(f"securities/{TICKER}")
card["description"][["name", "title", "value"]]

In [ ]:
# Посмотрите таблицу режимов торгов: boardid, title, market, is_primary, history_from, history_till

## Задание 2. Сколько строк вернул запрос · 10 минут
История торгов в основном режиме запрашивается по пути
`history/engines/stock/markets/shares/boards/TQBR/securities/{TICKER}` с параметрами `from` и `till` (даты в формате `ГГГГ-ММ-ДД`).

**Сначала предскажите:** сколько строк должно прийти за 2023 год? Запишите число до запуска.

Затем запросите 2023 год и посчитайте строки в таблице `history`. Посмотрите служебную таблицу `history.cursor`. Что означают её столбцы? Совпало ли число строк с предсказанием и почему?

**Предсказание:** …

**Ответ:** …

In [ ]:
path = f"history/engines/stock/markets/shares/boards/TQBR/securities/{TICKER}"
resp = iss_get(path, **{"from": "2023-01-01", "till": "2023-12-31"})
# Посчитайте строки в resp["history"] и посмотрите resp["history.cursor"]

## Задание 3. Загружаем всю историю · 25 минут
Допишите функцию `load_history`: она должна запрашивать страницы по очереди, увеличивая параметр `start`, пока не будут получены все строки (`TOTAL` из `history.cursor`). Готовый вариант есть в главе 2, но попробуйте сначала написать сами и разобраться в каждой строке.

Функция понадобится и для индекса: у индекса Мосбиржи `market="index"`, режим `board="SNDX"`.

Загрузите историю своей акции с 01.01.2021 по 31.10.2026 и проверьте результат ячейкой ниже: число строк по годам должно быть около 250, повторяющихся дат быть не должно.

In [ ]:
def load_history(ticker, start, end, board="TQBR", market="shares"):
    """Вся история торгов за период с учётом постраничной выдачи ISS."""
    path = f"history/engines/stock/markets/{market}/boards/{board}/securities/{ticker}"
    pages = []
    first = 0
    while True:
        resp = iss_get(path, **{"from": start, "till": end, "start": first})
        page = resp["history"]
        pages.append(page)
        total = ...          # всего строк: столбец TOTAL таблицы history.cursor
        first = ...          # с какой строки запрашивать следующую страницу
        if ...:              # когда остановиться
            break
    out = pd.concat(pages, ignore_index=True)
    out["TRADEDATE"] = pd.to_datetime(out["TRADEDATE"])
    return out

hist = load_history(TICKER, "2021-01-01", "2026-10-31")

In [ ]:
# Проверка загрузки
print("Строк:", len(hist))
print(hist.groupby(hist["TRADEDATE"].dt.year).size())
assert hist["TRADEDATE"].is_unique, "Есть повторяющиеся даты"
assert len(hist) > 1200, "Слишком мало строк: загружены не все страницы?"

## Задание 4. Паспорт данных · 15 минут
Заполните паспорт своей выгрузки:

| Поле | Значение |
|---|---|
| Источник и режим торгов | … |
| Дата выгрузки | … |
| Первая и последняя дата | … |
| Число строк | … |
| Строк без цены закрытия (`CLOSE` пустой) | … |
| Строк с нулевым объёмом (`VOLUME == 0`) | … |
| Дни торгов индекса IMOEX, которых нет у акции | … |

Затем ответьте:
1. На какие даты приходятся строки без цены закрытия? Что происходило на рынке в эти дни?
2. Что делать с такими строками перед расчётами: удалить, заполнить предыдущей ценой или оставить пустыми? От чего зависит выбор?
3. Почему отсутствие пустых ячеек в таблице ещё не доказывает, что ряд непрерывен?

**Ответ:** …

In [ ]:
imoex = load_history("IMOEX", "2021-01-01", "2026-10-31", board="SNDX", market="index")
# Найдите строки без CLOSE, строки с нулевым объёмом и даты IMOEX, которых нет у акции

In [ ]:
# График цены закрытия вашей акции

## Задание 5. Консолидация акций ВТБ · 20 минут
Загрузите историю **VTBR** в режиме TQBR за 2024 год. Посчитайте дневное изменение цены закрытия: `CLOSE / предыдущий CLOSE − 1`. Найдите день с самым большим по модулю изменением.

1. Что произошло в этот день? Подсказка – раздел «Сплиты и консолидации» главы 3. Во сколько раз изменилась цена и с каким коэффициентом это согласуется?
2. Что видно в данных в дни перед этой датой?
3. Скорректируйте ряд так, чтобы цены до события были сопоставимы с ценами после. Постройте исходный и скорректированный ряды на одном графике в логарифмическом масштабе (`plt.yscale("log")`).
4. Каким получился бы «рост цены за 2024 год» по исходному ряду и по скорректированному?

**Ответ:** …

In [ ]:
vtb = load_history("VTBR", "2024-01-01", "2024-12-31")
# Дневное изменение, самый большой скачок, корректировка и график

## Задание 6. Дивидендный разрыв · 15 минут
Сбербанк выплатил дивиденд 34,84 руб. на обыкновенную акцию за 2024 год. Дата закрытия реестра – 18.07.2025. При расчётах T+1 последний день, когда можно купить акцию с правом на дивиденд, – 17.07.2025.

Загрузите SBER и IMOEX за 16–21 июля 2025 г. Сравните цену закрытия 17 июля с ценой открытия 18 июля (`OPEN`) для акции и для индекса.

1. На сколько рублей и процентов открылась ниже акция? А индекс?
2. Сопоставьте падение с размером дивиденда, в том числе после налога 13%. Почему они не совпадают в точности?
3. Как такой разрыв исказит доходность, если считать её по `CLOSE`? Как эту проблему решает скорректированная цена (глава 3)?

**Ответ:** …

In [ ]:
sber = load_history("SBER", "2025-07-16", "2025-07-21")
imoex_jul = load_history("IMOEX", "2025-07-16", "2025-07-21", board="SNDX", market="index")
# Сравните CLOSE 17.07 и OPEN 18.07

## Задание 7. Самые большие движения вашей акции · 15 минут
Вернитесь к своей акции. Найдите **три дня** с самым большим по модулю дневным изменением `CLOSE` за 2021–2026 гг. Вспомните строки без цены из задания 4: какое изменение получится в первый день после них и не потеряется ли этот день при расчёте? Для каждого дня:
- посмотрите изменение индекса IMOEX в тот же день;
- определите, на что это похоже: событие всего рынка, сплит или консолидация, дивидендный разрыв, новость компании, ошибка данных;
- если это событие компании, найдите подтверждение (сайт компании, раскрытие на e-disclosure.ru, новости) и запишите источник.

**Ответ:**

| Дата | Изменение акции | Изменение IMOEX | Что это | Источник |
|---|---|---|---|---|
| … | … | … | … | … |

In [ ]:
# Три самых больших движения и изменение индекса в те же дни

## Задание 8. Вывод и журнал ИИ · 15 минут
Напишите 150–250 слов:
- паспорт ваших данных в двух-трёх предложениях;
- какие три проблемы данных вы нашли сегодня (на своей акции, ВТБ, Сбербанке) и что с каждой нужно сделать, прежде чем считать доходность;
- что осталось непонятным.

**Проверка ИИ.** Попросите ИИ-ассистента написать функцию загрузки дневной истории акции с Московской биржи за 5 лет. Проверьте его код: учитывает ли он постраничную выдачу? Какой режим торгов использует? Что делает со строками без цены? Если ИИ не используете, так и напишите.

**Вывод:** …

**Журнал ИИ:** запрос … ответ ассистента (кратко) … что проверили и что нашли …

## Необязательно: сверка с независимым источником
Сравните цены закрытия SBER из ISS с файлом первого занятия (`prices_2021_2026.csv`, столбец `SBER_close`). В скольких днях цены расходятся больше чем на копейку? Что общего у этих дат? Посмотрите, совпадает ли значение из файла с другим полем ISS, например `LEGALCLOSEPRICE`.

## Перед сдачей
Перезапустите ядро и выполните все ячейки сверху вниз. Проверьте, что во всех ячейках «Ответ» есть текст. Сохраните файл как `selfstudy2_фамилия.ipynb` и загрузите его через [Google-форму](https://forms.gle/sP7LxYF9y8hMVgPK7). Не включайте пароли, токены и личные сведения.